In [ ]:
import json
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pymutspec.annotation import rev_comp
from pymutspec.constants import possible_sbs192, possible_codons
from pymutspec.draw import plot_mutspec192
import SigProfilerAssignment
from scipy.stats import spearmanr

from utils import sbs2effect, collapse_sbs192, complete_sbs_columns

sns.set_style()
%matplotlib inline

In [ ]:
# version of SigProfilerAssignment
SigProfilerAssignment.__version__

In [ ]:
# funcs for specific data processing and writing


def read_human_triplet_counts(include_lower_nucls=False):
    with open("./data/triplet_counts_GRCh37.json") as fin:
        _human_counts_raw = json.load(fin)
        if include_lower_nucls:
            human_counts = defaultdict(int)
            for triplet, n in _human_counts_raw.items():
                human_counts[triplet.upper()] += n
            human_counts = dict(human_counts)        
        else:
            # without aAa, tAA, tgc etc. Only upper case
            _human_counts_raw = {x: _human_counts_raw[x] for x in possible_codons}
            human_counts = defaultdict(int)
            for triplet, n in _human_counts_raw.items():
                human_counts[triplet.upper()] += n
            human_counts = dict(human_counts)
        return human_counts


def save_wide_cls_spectra(df: pd.DataFrame, filename=None, human_counts: dict={}, scale_coef=6.6e-5):
    assert df.shape[1] == 96
    df.columns.name = "MutationType"
    multiplier = df.columns.to_series().apply(lambda x: x[0] + x[2] + x[-1]).map(human_counts)
    rescaled = (df * multiplier * scale_coef).T
    rescaled = rescaled.round().astype(int)
    if filename is not None:
        rescaled.sort_index().to_csv(filename, sep="\t")
    return rescaled

In [ ]:
possible_sbs6 = ["C>A", "C>G", "C>T", "T>A", "T>C", "T>G"]
possible_sbs96 = [x for x in possible_sbs192 if x[2:5] in possible_sbs6]
print(possible_sbs96)

## Load and Explore human trinucl counts for renormalization (emulation of human spectra for COSMIC decomposition) 

In [ ]:
# read human genome trinucleotide counts
human_counts = read_human_triplet_counts()
human_counts_wl = read_human_triplet_counts(True)

In [ ]:
x = pd.Series(human_counts).sort_index().values
y = pd.Series(human_counts_wl).sort_index().values
print(spearmanr(x, y))
# plt.scatter(x, y)
# plt.show()

In [ ]:
# check forvard and rev-comp triplets freqs
d = defaultdict(dict)
for triplet, n in human_counts_wl.items():
    if triplet[1] in "CT":
        # print(triplet, n)
        d[triplet]["cosmic"] = n
    else:
        d[rev_comp(triplet)]["other"] = n

d = pd.DataFrame(d).T
d["diff"] = d.cosmic - d.other
d["diff%"] = d["diff"] / d['cosmic']
d

Forvard and rev-comp triplets freqs are almost equal, so we need to use triplets with C & T in 2nd position

## Need to generate many variants of 96-comp spectra:

- **low ts**: only transitions (Gh>Ah and Th>Ch)
- **low ts&tv**: transitions (Gh>Ah and Th>Ch) & averaged transversions [(h + l) / 2]
- **high ts**: only transitions (Ch>Th and Ah>Gh)
- **high ts&tv**: transitions (Ch>Th and Ah>Gh) & averaged transversions [(h + l) / 2]

- **diff**: **high ts** - **low ts**
- **diff&tv**: **high ts&tv** - **low ts&tv**

In [ ]:
# Load Gobiiformes species-level spectra (long format, already H-strand)
species_spectra = pd.read_csv('./data/gobiiformes_spectra_long.csv')
# Data is already in H-strand notation — NO rev_comp needed
# (The original notebook used rev_comp because its input was L-strand)

# Filter to families with >=3 species for robust mean spectra
fam_counts = species_spectra.groupby('family')['Species'].nunique()
valid_families = fam_counts[fam_counts >= 3].index.tolist()
species_spectra = species_spectra[species_spectra['family'].isin(valid_families)]

print(f"Total species: {species_spectra['Species'].nunique()}")
print(f"Families (>=3 species): {valid_families}")
species_spectra

In [ ]:
# check most popular sbs - we see C>T, therefore we are using H-strand 
species_spectra.groupby('Mut').MutSpec.mean().sort_values()

In [ ]:
species_spectra_wide = complete_sbs_columns(
    species_spectra.groupby(["family", "Species", "Mut"]).MutSpec.sum().unstack(), 192)
cls_spectra_wide = species_spectra_wide.mean(level=0)
cls_spectra_wide.head()

### Get spectra that contain only transitions (**Only Ts's**)

In [ ]:
# get spectra with "low Ts" (minor transitions in the complementar pairs) only
sbs_low_ts = ["G>A", "T>C"]
low_ts_only = collapse_sbs192(complete_sbs_columns(
    cls_spectra_wide[[x for x in possible_sbs192 if x[2:5] in sbs_low_ts]], 192), 96)
low_ts_only

In [ ]:
# same with "high Ts" (major transitions in the complementar pairs)
sbs_high_ts = ["C>T", "A>G"]
high_ts_only = collapse_sbs192(complete_sbs_columns(
    cls_spectra_wide[[x for x in possible_sbs192 if x[2:5] in sbs_high_ts]], 192), 96)
high_ts_only

In [ ]:
# substract "low Ts" from "high Ts" to get "diff"
diff_ts_only = high_ts_only - low_ts_only
diff_ts_only[diff_ts_only < 0] = 0

### Get spectra that contain transitions and transversions (**Ts&Tv**)

In [ ]:
# average number of transversions in each complementary pair
sbs_tv = ['A>C', 'A>T', 'C>A', 'C>G', 'G>C', 'G>T', 'T>A', 'T>G']
tv_only = collapse_sbs192(complete_sbs_columns(
    cls_spectra_wide[[x for x in possible_sbs192 if x[2:5] in sbs_tv]], 192), 96) / 2
tv_only

In [ ]:
low_ts_with_tv  = tv_only + low_ts_only
high_ts_with_tv = tv_only + high_ts_only
diff_ts_with_tv = tv_only + diff_ts_only

low_ts_with_tv

In [ ]:
# Concatenate all 6 variants pairwisely and save to input directory

SAMPLE_OUTDIR = './data/SigProfilerAssignment/input/'

# we can use any subsample of human genome to calculate trinucl freqs and we will give 
# same results due to too big counts of trinucleotides (data not shown but precicely checked)
include_lower_nucls = False
THR = 0.000044 if include_lower_nucls else 0.000066
human_counts = read_human_triplet_counts(include_lower_nucls)

d = []
for x, _lbl in zip([low_ts_only, low_ts_with_tv], ["Ts only", "Ts & Tv"]):
    x = x.copy()
    x.index = [y + f"__{_lbl}" for y in x.index]
    d.append(x)
    
total_samples = pd.concat(d)
save_wide_cls_spectra( 
    total_samples, SAMPLE_OUTDIR+"low_Ts_samples.txt", human_counts, THR,
)


d = []
for x, _lbl in zip([high_ts_only, high_ts_with_tv], ["Ts only", "Ts & Tv"]):
    x = x.copy()
    x.index = [y + f"__{_lbl}" for y in x.index]
    d.append(x)
    
total_samples = pd.concat(d)
save_wide_cls_spectra( 
    total_samples, SAMPLE_OUTDIR+"high_Ts_samples.txt", human_counts, THR,
)


d = []
for x, _lbl in zip([diff_ts_only, diff_ts_with_tv], ["Ts only", "Ts & Tv"]):
    x = x.copy()
    x.index = [y + f"__{_lbl}" for y in x.index]
    d.append(x)
    
total_samples = pd.concat(d)
save_wide_cls_spectra( 
    total_samples, SAMPLE_OUTDIR+"high_minus_low_Ts_samples.txt", human_counts, THR,
)

### Check input spectra
These spectra renormalized, e.g. we can imagine we translated observed mutagenesis from vertebrates classes to human genome

So, currently we can run signatures assignemnt on COSMIC

In [ ]:
low = pd.read_csv('./data/SigProfilerAssignment/input/low_Ts_samples.txt', sep='\t', index_col=0)
high = pd.read_csv('./data/SigProfilerAssignment/input/high_Ts_samples.txt', sep='\t', index_col=0)
hml = pd.read_csv('./data/SigProfilerAssignment/input/high_minus_low_Ts_samples.txt', sep='\t', index_col=0)

In [ ]:
x = high['Gobiidae__Ts & Tv'].rename('MutSpec')
x.index.name = 'Mut'
x = x.reset_index()
plot_mutspec192(x, title='Gobiidae High Ts&Tv', figsize=(15, 5));

In [ ]:
x = low['Gobiidae__Ts & Tv'].rename('MutSpec')
x.index.name = 'Mut'
x = x.reset_index()
plot_mutspec192(x, title='Gobiidae Low Ts&Tv', figsize=(15, 5));

## Run SigProfilerAssignment

In [ ]:
# Run sigprofiller cosmic_fit func to decompose chordates classes spcectra to COSMIC signatures
# Run on several different spectra types (Low, High, Diff) separately

from SigProfilerAssignment import Analyzer as Analyze

samples_pattern = "./data/SigProfilerAssignment/input/{}_samples.txt"
OUTDIR = "./data/SigProfilerAssignment/output/"
output_pattern = OUTDIR + "{}/"

exclude_signature_subgroups = [
    'Artifact_signatures',
    'Immunosuppressants_signatures',
    'Treatment_signatures',
    'Lymphoid_signatures',
    'Colibactin_signatures',
    'AA_signatures',
]

for label in ["low_Ts", "high_Ts", "high_minus_low_Ts"]:
    samples = samples_pattern.format(label)
    output  = output_pattern.format(label)
    Analyze.cosmic_fit(samples, output, input_type="matrix", context_type="96",
                    cosmic_version=3.3, exome=False,
                    nnls_remove_penalty=0.01, nnls_add_penalty = 0.02,
                    exclude_signature_subgroups=exclude_signature_subgroups, 
                    export_probabilities=False, make_plots=True,
                    sample_reconstruction_plots=False, verbose=False)

In [ ]:
# make Solution_Stats with assignment quality
d = pd.concat([pd.read_csv(f"{OUTDIR}/{label}/"
        "Assignment_Solution/Solution_Stats/Assignment_Solution_Samples_Stats.txt", sep='\t')\
            .assign(Run=label).set_index(['Run', 'Sample Names']) \
            for label in ["low_Ts", "high_Ts", "high_minus_low_Ts"]])

d.to_csv(f'{OUTDIR}/Solution_Stats.txt', sep='\t')
d.head()

## Plot beautiful images and add them to single panel
see modified scpipt [plotActivity.py](./plotActivity.py) from sigProfilerPlotting lib

In [ ]:
from pandas import CategoricalDtype
import re

### 1. Gobiiformes families — ordered by descending species count
families = ['Gobiidae', 'Eleotridae', 'Odontobutidae']  # families with >=3 species

# 生成排序用的类别顺序：high-{fam}, low-{fam}, diff-{fam}
set_order = []
for fam in families:
    set_order.extend([f'high-{fam}', f'low-{fam}', f'diff-{fam}'])

cat_type = CategoricalDtype(categories=set_order, ordered=True)

pt_del = r'__Ts & Tv|__Ts'   # 用于清理索引文本

### 2. 动态生成 index_odr 列表
index_odr = []
for fam in families:
    index_odr.append(f'{fam}__Ts only')
for fam in families:
    index_odr.append(f'{fam}__Ts & Tv')

### 3. 读取并合并三个频谱类型的文件
data = []
for label in ["high_Ts", "low_Ts", "high_minus_low_Ts"]:
    d = pd.read_csv(f'{OUTDIR}/{label}/'
                'Assignment_Solution/Activities/Assignment_Solution_Activities.txt', 
                sep='\t', index_col=0).loc[index_odr]
    
    if label == 'low_Ts':
        label_prefix = 'low-'
    elif label == 'high_Ts':
        label_prefix = 'high-'
    elif label == 'high_minus_low_Ts':
        label_prefix = 'diff-'
    
    d.index = label_prefix + d.index.str.replace(' only', '')
    data.append(d)

df_full = pd.concat(data)

# 排序与清理
df_full['Samples_Sort'] = df_full.index.str.replace(pt_del, '', regex=True)
df_full['TsTv'] = df_full.index.str.contains('Tv')
df_full['Samples_Sort'] = df_full['Samples_Sort'].astype(cat_type)
df_full = df_full.sort_values(['Samples_Sort', 'TsTv']).drop(['Samples_Sort', 'TsTv'], axis=1)

# 提取仅转换的谱
df_ts_only = df_full.loc[~df_full.index.str.contains('Tv')]
df_ts_only.index = df_ts_only.index.str.replace('__Ts', '')

# 保存
path_to_total_acivities = f'{OUTDIR}Assignment_Solution_Activities.txt'
df_full.to_csv(path_to_total_acivities, sep='\t')
df_ts_only.to_csv(path_to_total_acivities.replace('.txt', '_Ts.txt'), sep='\t')
df_full

In [ ]:
# total
from plotActivity import plotActivity

# 保存 PDF
plotActivity(
    path_to_total_acivities, f"{OUTDIR}/total.pdf", 
    bin_size=30, 
    delimiter_step=6, delimiter_size=1,
)

# 保存 PNG
plotActivity(
    path_to_total_acivities, f"{OUTDIR}/total.png", 
    bin_size=30, 
    delimiter_step=6, delimiter_size=1,
)

In [ ]:
# only Ts
outpath = f"{OUTDIR}/only_Ts.pdf"

plotActivity(
    path_to_total_acivities.replace('.txt', '_Ts.txt'), 
    outpath, 
    bin_size=50,
    delimiter_step=3, delimiter_size=1,
    rename=True, figure_width=4,
)